# S6E10 · RealMLP + original-dataset statistics + nearest original neighbours

**Credit:** preprocessing and RealMLP hyper-parameters follow
[PS|S6|E10: RealMLP · PyTabKit](https://www.kaggle.com/code/yekenot/ps-s6-e10-realmlp-pytabkit) by Vladimir Demidov:
every numeric column is also used as a categorical, count encoding, four interaction keys with in-fold target
encoding, and a tuned RealMLP configuration.

**What I add:** satisfaction rates looked up in the public CC0 dataset this competition was synthesised from
(per column value, and per exact rating vector / service profile). Those rows are disjoint from the competition
data, so the lookup cannot leak competition labels. Same 5-fold `StratifiedKFold(seed=42)` as my tree models,
so the OOF predictions can be blended with them.


This version only changes the RealMLP seed (`random_state=2024`) for seed bagging.


In [ ]:
!pip install -q pytabkit
import glob, os, time, warnings
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from pytabkit import RealMLP_TD_Classifier
warnings.filterwarnings("ignore")

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, name
    return hits[0]

train = pd.read_csv(find("train.csv")); test = pd.read_csv(find("test.csv"))
orig = pd.read_csv(find("data.csv"))
TARGET = "satisfaction"
y = train.pop(TARGET).astype(int).to_numpy()
orig[TARGET] = orig[TARGET].astype(int)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(train.shape, test.shape, orig.shape, DEVICE)


## Original-dataset statistics (my addition)

In [ ]:
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
RATINGS = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness']
RATING_KEY = RATINGS
SERVICE_KEY = ["Customer Type", "Type of Travel", "Class"] + RATINGS

def original_stats(df: pd.DataFrame, orig: pd.DataFrame, smoothing: float = 20.0) -> pd.DataFrame:
    """Target statistics from the original dataset, looked up for each competition row.

    The original rows are disjoint from the competition data, so these features cannot leak the
    competition labels and are safe to compute once, outside the CV loop.
    """
    prior = orig[TARGET].mean()
    out = pd.DataFrame(index=df.index)
    raw_cols = CATEGORICAL + RATINGS + ["Age", "Flight Distance", "Departure Delay in Minutes"]
    for col in raw_cols:
        g = orig.groupby(col)[TARGET].agg(["sum", "count"])
        te = (g["sum"] + smoothing * prior) / (g["count"] + smoothing)
        out[f"orig_te_{col}"] = df[col].map(te).astype(float).fillna(prior)
    for name, key in [("ratings", RATING_KEY), ("service", SERVICE_KEY), ("full", [c for c in orig.columns if c != TARGET])]:
        g = orig.groupby(key, dropna=False)[TARGET].agg(["mean", "count"]).reset_index()
        g = g.rename(columns={"mean": f"orig_key_{name}_mean", "count": f"orig_key_{name}_n"})
        merged = df[key].merge(g, on=key, how="left")
        out[f"orig_key_{name}_mean"] = merged[f"orig_key_{name}_mean"].to_numpy()
        out[f"orig_key_{name}_n"] = merged[f"orig_key_{name}_n"].fillna(0).to_numpy()
    return out

O_tr, O_te = original_stats(train, orig), original_stats(test, orig)
O_tr, O_te = O_tr.drop(columns=["orig_key_full_mean", "orig_key_full_n"]), O_te.drop(columns=["orig_key_full_mean", "orig_key_full_n"])
O_med = O_tr.median()
O_tr, O_te = O_tr.fillna(O_med).astype("float32"), O_te.fillna(O_med).astype("float32")
print(O_tr.shape)


In [ ]:
def _matrix(df: pd.DataFrame) -> np.ndarray:
    parts = [df[RATINGS].to_numpy(float)]
    for col in CATEGORICAL:
        parts.append(pd.get_dummies(df[col]).reindex(columns=sorted(df[col].unique()), fill_value=0).to_numpy(float) * 2.0)
    parts.append(df[["Age"]].to_numpy(float) / 10.0)
    parts.append(np.log1p(df[["Flight Distance"]].to_numpy(float)))
    return np.hstack(parts)

from sklearn.neighbors import NearestNeighbors
def knn_block(df, orig_df, ks=(5, 25, 100)):
    cats = {c: sorted(orig_df[c].unique()) for c in CATEGORICAL}
    cast = lambda d: d.assign(**{c: pd.Categorical(d[c], categories=cats[c]) for c in CATEGORICAL})
    o = cast(orig_df.copy()); yo = o[TARGET].to_numpy()
    nn = NearestNeighbors(n_neighbors=max(ks), n_jobs=-1).fit(_matrix(o))
    dist, idx = nn.kneighbors(_matrix(cast(df.copy())))
    out = {"knn_dist1": dist[:, 0], "knn_dist25": dist[:, 24]}
    for k in ks:
        out[f"knn_mean{k}"] = yo[idx[:, :k]].mean(axis=1)
    w = 1.0 / (dist[:, :25] + 0.5)
    out["knn_wmean25"] = (yo[idx[:, :25]] * w).sum(axis=1) / w.sum(axis=1)
    return pd.DataFrame(out, index=df.index).astype("float32")

t = time.time()
K_tr, K_te = knn_block(train, orig), knn_block(test, orig)
O_tr, O_te = pd.concat([O_tr, K_tr], axis=1), pd.concat([O_te, K_te], axis=1)
print("knn", round(time.time() - t), "s", O_tr.shape)


## Preprocessing (public recipe)

In [ ]:
X, X_test = train.drop(columns=["id"]), test.drop(columns=["id"])
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()
COMBOS = [("Class", "Type of Travel", "Gender"), ("Class", "Type of Travel", "Age"),
          ("Food and drink", "Cleanliness"), ("Flight Distance", "Departure/Arrival time convenient")]
combo_names = ["_".join(c) + "_" for c in COMBOS]

def engineer(df, ref):
    """ref is the training frame: category codes and counts are learned from it only."""
    out = df.copy()
    for c in cat_cols:
        out[c] = out[c].fillna("missing")
    for c in num_cols:
        out[c] = out[c].fillna(0.0)
    for c in num_cols:
        uniques = pd.Index(ref[c].fillna(0.0).unique())
        out[f"{c}_cat_"] = pd.Categorical(uniques.get_indexer(out[c]))
    for c in cat_cols + ["Leg room service"]:
        out[f"_{c}_count"] = out[c].map(ref[c].fillna("missing").value_counts()).fillna(0).astype("int32")
    for cols, name in zip(COMBOS, combo_names):
        key = lambda d: d[list(cols)].fillna(0).astype(str).agg("_".join, axis=1)
        uniques = pd.Index(key(ref).unique())
        out[name] = pd.Categorical(uniques.get_indexer(key(out)))
    return out

Xe, Xe_test = engineer(X, X), engineer(X_test, X)
Xe, Xe_test = pd.concat([Xe, O_tr], axis=1), pd.concat([Xe_test, O_te], axis=1)
print(Xe.shape, Xe_test.shape)


## RealMLP configuration (public recipe)

In [ ]:
params = dict(
    random_state=2024, verbosity=0, val_metric_name="1-auc_ovr", device=DEVICE,
    n_ens=8, n_epochs=3, batch_size=288, use_early_stopping=False,
    lr=0.053, wd=0.015, sq_mom=0.988, lr_sched="flat_anneal", wd_sched="cos_log_15", first_layer_lr_factor=0.25,
    embedding_size=5, max_one_hot_cat_size=2, hidden_sizes=[512, 256, 128], act="silu", p_drop=0.05, p_drop_sched="pow5",
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name="gelu", plr_lr_factor=0.1151, plr_sigma=2.33,
    ls_eps=0.01, ls_eps_sched="sqrt_cos", add_front_scale=False, bias_init_mode="neg-uniform-dynamic-2",
    tfms=["one_hot", "median_center", "robust_scale", "smooth_clip", "embedding", "l2_normalize"],
)


## 5-fold CV with in-fold target encoding

In [ ]:
oof, test_pred = np.zeros(len(y)), np.zeros(len(Xe_test))
drop_after_te = ["Class_Type of Travel_Age_", "Flight Distance_Departure/Arrival time convenient_"]
for k, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(Xe, y)):
    t = time.time()
    X_tr, X_va, X_te = Xe.iloc[tr].copy(), Xe.iloc[va].copy(), Xe_test.copy()
    enc = TargetEncoder(cv=5, smooth="auto", shuffle=True, random_state=42)  # training fold only
    names = [f"_{c}TE" for c in combo_names]
    X_tr[names] = enc.fit_transform(X_tr[combo_names].astype(str), y[tr])
    X_va[names] = enc.transform(X_va[combo_names].astype(str))
    X_te[names] = enc.transform(X_te[combo_names].astype(str))
    X_tr, X_va, X_te = (d.drop(columns=drop_after_te) for d in (X_tr, X_va, X_te))
    model = RealMLP_TD_Classifier(**params)
    model.fit(X_tr, y[tr], X_va, y[va])
    oof[va] = model.predict_proba(X_va)[:, 1]
    test_pred += model.predict_proba(X_te)[:, 1] / 5
    print(f"fold {k}: AUC {roc_auc_score(y[va], oof[va]):.6f}  ({time.time()-t:.0f}s)", flush=True)
    torch.cuda.empty_cache()
print(f"OOF AUC {roc_auc_score(y, oof):.6f}")
np.save("realmlp_v4_oof.npy", oof); np.save("realmlp_v4_test.npy", test_pred)
pd.DataFrame({"id": test["id"], TARGET: test_pred}).to_csv("submission.csv", index=False, float_format="%.6f")
